# 02 - Classical Machine Learning Models
## Predicting Human Intestinal Absorption (HIA)
**Student:** Mohammad Karamath Fardeen | ID: 25251265  
**Supervisor:** Kolawole Adebayo | Maynooth University | 2025-2026

Models: Logistic Regression, Random Forest, XGBoost, LightGBM

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib, os

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (roc_auc_score, f1_score, accuracy_score,
                             matthews_corrcoef, roc_curve, confusion_matrix)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from imblearn.over_sampling import SMOTE

SEED = 42
FEATURE_COLS = ['MolWt','LogP','TPSA','HBD','HBA','RotBonds',
                'RingCount','AromaticRings','HeavyAtoms',
                'FractionCSP3','MolMR','NumHeteroatoms']
os.makedirs('results/metrics', exist_ok=True)
os.makedirs('results/figures', exist_ok=True)
print('Setup complete!')

## 1. Load Data and Apply SMOTE

In [ ]:
train = pd.read_csv('data/processed/hia_train_features.csv').dropna(subset=FEATURE_COLS)
valid = pd.read_csv('data/processed/hia_valid_features.csv').dropna(subset=FEATURE_COLS)
test  = pd.read_csv('data/processed/hia_test_features.csv').dropna(subset=FEATURE_COLS)

X_train, y_train = train[FEATURE_COLS].values, train['Y'].values
X_valid, y_valid = valid[FEATURE_COLS].values, valid['Y'].values
X_test,  y_test  = test[FEATURE_COLS].values,  test['Y'].values

# Scale features
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_valid_sc = scaler.transform(X_valid)
X_test_sc  = scaler.transform(X_test)
joblib.dump(scaler, 'results/metrics/scaler.joblib')

# Apply SMOTE to training set only
sm = SMOTE(random_state=SEED)
X_train_res, y_train_res = sm.fit_resample(X_train_sc, y_train)
print(f'Before SMOTE: {dict(zip(*np.unique(y_train, return_counts=True)))}')
print(f'After SMOTE:  {dict(zip(*np.unique(y_train_res, return_counts=True)))}')

## 2. Define and Train Models

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, C=1.0, random_state=SEED, class_weight='balanced'),
    'Random Forest':       RandomForestClassifier(n_estimators=300, random_state=SEED, class_weight='balanced', n_jobs=-1),
    'XGBoost':             XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                                         subsample=0.8, colsample_bytree=0.8,
                                         eval_metric='logloss', random_state=SEED, n_jobs=-1),
    'LightGBM':            LGBMClassifier(n_estimators=300, learning_rate=0.05,
                                          class_weight='balanced', random_state=SEED, n_jobs=-1, verbose=-1),
}

results = []
for name, model in models.items():
    print(f'Training: {name}...')
    model.fit(X_train_res, y_train_res)
    joblib.dump(model, f'results/metrics/{name.replace(" ","_").lower()}.joblib')
    
    for split_name, X, y in [('valid', X_valid_sc, y_valid), ('test', X_test_sc, y_test)]:
        y_pred  = model.predict(X)
        y_proba = model.predict_proba(X)[:, 1]
        results.append({
            'model': name, 'split': split_name,
            'roc_auc':  round(roc_auc_score(y, y_proba), 4),
            'f1':       round(f1_score(y, y_pred), 4),
            'accuracy': round(accuracy_score(y, y_pred), 4),
            'mcc':      round(matthews_corrcoef(y, y_pred), 4),
        })

results_df = pd.DataFrame(results)
results_df.to_csv('results/metrics/classical_results.csv', index=False)
print('\n--- Test Set Results ---')
print(results_df[results_df['split']=='test'].to_string(index=False))

## 3. ROC Curves

In [ ]:
plt.figure(figsize=(8, 6))
colors = ['#3498DB', '#9B59B6', '#E67E22', '#27AE60']

for (name, model), color in zip(models.items(), colors):
    y_proba = model.predict_proba(X_test_sc)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = roc_auc_score(y_test, y_proba)
    plt.plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})', linewidth=2, color=color)

plt.plot([0,1],[0,1],'k--', linewidth=1, label='Random')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves — Classical ML Models (Test Set)', fontweight='bold')
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig('results/figures/roc_curves_classical.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Confusion Matrices

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 9))
axes = axes.flatten()

for ax, (name, model) in zip(axes, models.items()):
    y_pred = model.predict(X_test_sc)
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax, cbar=False,
                xticklabels=['Low (0)', 'High (1)'],
                yticklabels=['Low (0)', 'High (1)'],
                annot_kws={'size': 14, 'weight': 'bold'})
    ax.set_title(name, fontweight='bold')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')

plt.suptitle('Confusion Matrices — Classical ML Models (Test Set)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('results/figures/confusion_matrices_classical.png', dpi=150, bbox_inches='tight')
plt.show()